# ML Lab · Module 5, Unit 9: Boosting and SHAP

**Prof. Dr. Mehmet Songur** · ML Lab, Module 5

Data: 30,000 credit card customers of a bank in Taiwan (UCI, Yeh and Lien 2009, CC BY 4.0). The setup cell downloads a table called `credit` and installs the `shap` library if needed. The target `default`: default in October 2005 (1). Trees can use the payment codes (`pay_0` … `pay_6`) in raw form; the protected attributes (`sex`, `marriage`) are left out. **Have Gemini write the code**, then **audit it**.

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running.
3. If a check cell passes, it gives you a **completion code**; paste it into the box in ML Lab.

In [ ]:
#@title Setup — run it, do not change it

import base64, subprocess, sys, warnings
try:
    import shap
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

credit = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/credit/taiwan_default.csv")
print("credit:", credit.shape, "| default rate:", round(credit["default"].mean(), 3), "| shap", shap.__version__)
_E, _T = train_test_split(credit.copy(), test_size=0.3, random_state=0, stratify=credit["default"])
_COLS = [c for c in credit.columns if c not in ("default", "sex", "marriage")]
_CAND = [100, 200, 400, 800]

def _k(s):
    return base64.b64decode(s).decode()

def _hgb(n):
    return HistGradientBoostingClassifier(max_iter=n, learning_rate=0.05, max_leaf_nodes=8, early_stopping=False, random_state=1)

def check_A(train, test, X_tr, X_te, n_chosen, model, p_test, auc_test):
    if len(test) != len(_T) or set(test.index) != set(_T.index) or set(train.index) != set(_E.index):
        print("❌ The split differs from the expected one: train_test_split(credit, test_size=0.3, random_state=0, stratify=credit['default'])."); return
    if {"sex", "marriage"} & set(X_tr.columns):
        print("❌ Protected attributes (sex, marriage) are in the model. Leave them out."); return
    if "default" in X_tr.columns:
        print("❌ The target ('default') is among the features: target leakage."); return
    if set(X_tr.columns) != set(_COLS) or list(X_te.columns) != list(X_tr.columns):
        print("❌ The features differ from the expected ones: every raw column except default, sex and marriage, in the same order in training and test."); return
    if not isinstance(model, HistGradientBoostingClassifier):
        print("❌ model should be a HistGradientBoostingClassifier."); return
    if model.early_stopping is not False:
        print("❌ early_stopping is on: the model chooses the number of trees itself with a part it sets aside from the training data. Set early_stopping=False."); return
    if model.learning_rate != 0.05 or model.max_leaf_nodes != 8:
        print("❌ The settings differ from the expected ones: learning_rate=0.05, max_leaf_nodes=8."); return
    print("Check: cross-validating the candidate numbers of trees on the training data (a few seconds)…")
    cv = {n: cross_val_score(_hgb(n), X_tr, train["default"], cv=5, scoring="roc_auc").mean() for n in _CAND}
    ref = max(cv, key=cv.get)
    if int(n_chosen) != ref:
        tst = {n: roc_auc_score(test["default"], _hgb(n).fit(X_tr, train["default"]).predict_proba(X_te)[:, 1]) for n in _CAND}
        if int(n_chosen) == max(tst, key=tst.get):
            print(f"❌ The number of trees ({n_chosen}) was chosen on the test data. Choose it on the training data with cross_val_score(..., cv=5, scoring='roc_auc')."); return
        print(f"❌ The chosen number of trees ({n_chosen}) differs from the expected one. Candidates {_CAND}, 5-fold cross-validation on the training data, metric roc_auc."); return
    if model.max_iter != int(n_chosen):
        print("❌ model was not fitted with the chosen number of trees (max_iter=n_chosen)."); return
    p = np.asarray(p_test, dtype=float)
    if len(p) != len(test) or len(np.unique(p)) <= 2:
        print("❌ p_test is not a probability. Use predict_proba(...)[:, 1] instead of predict."); return
    if not np.allclose(p, model.predict_proba(X_te)[:, 1]):
        print("❌ p_test does not match the model's default probabilities for X_te."); return
    a = roc_auc_score(test["default"], p)
    if abs(float(auc_test) - a) > 1e-6:
        print("❌ auc_test does not match roc_auc_score(test['default'], p_test)."); return
    print(f"✅ All checks passed (number of trees {ref}, test AUC ≈ {a:.3f}; Module 4's logistic regression 0.762). Your completion code: " + _k("TUw1LUJTVC02UVJN"))

def check_B(model, X_te, sv, base, importance):
    Xs = X_te.iloc[:2000]
    sv = np.asarray(sv)
    if sv.shape != Xs.shape:
        print(f"❌ sv has shape {sv.shape}; for the first 2,000 test customers it should be {Xs.shape} (X_te.iloc[:2000])."); return
    gap = np.abs(float(base) + sv.sum(1) - model.decision_function(Xs)).max()
    if gap > 1e-6:
        print(f"❌ Additivity does not hold (largest gap {gap:.2g}): base + sv.sum(1) should equal model.decision_function(X_te.iloc[:2000]). For base use shap.TreeExplainer(model).expected_value."); return
    ref = shap.TreeExplainer(model).shap_values(Xs)
    if not np.allclose(sv, ref, atol=1e-6):
        print("❌ sv does not match shap.TreeExplainer(model).shap_values(X_te.iloc[:2000])."); return
    if not isinstance(importance, pd.Series) or set(importance.index) != set(Xs.columns):
        print("❌ importance should be a pandas Series with the column names in its index."); return
    expected = pd.Series(np.abs(sv).mean(0), index=Xs.columns)
    if not np.allclose(importance.reindex(expected.index).values, expected.values):
        print("❌ importance should be each column's mean |SHAP|: np.abs(sv).mean(axis=0)."); return
    if not importance.is_monotonic_decreasing:
        print("❌ importance should be sorted from largest to smallest (sort_values(ascending=False))."); return
    print(f"✅ All checks passed (most important variable {importance.index[0]}, mean |SHAP| {importance.iloc[0]:.2f}). Your completion code: " + _k("TUw1LVNIUC05WEtE"))

credit.head(3)

## Task A — Boosting

1. Split `credit` into `train` and `test` with `train_test_split(credit, test_size=0.3, random_state=0, stratify=credit["default"])`.
2. Features: `X_tr` and `X_te`, every column except `default`, `sex` and `marriage` (in raw form).
3. For `HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=8, early_stopping=False, random_state=1)`, compare the `max_iter` candidates `[100, 200, 400, 800]` by the mean of `cross_val_score(..., cv=5, scoring="roc_auc")` **on the training data only**; the best is `n_chosen`.
4. Fit the model with this value on all of the training data: `model`. Test probabilities `p_test`, test AUC `auc_test`. Compare with Module 4's logistic regression, 0.762.

**Audit questions:** Are sex and marriage left out? Was the number of trees chosen with the test data? Is `early_stopping=False`?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it (it may take about half a minute)

In [ ]:
check_A(train, test, X_tr, X_te, n_chosen, model, p_test, auc_test)

## Task B — SHAP

1. With `ex = shap.TreeExplainer(model)` compute the SHAP values of the first 2,000 test customers: `sv = ex.shap_values(X_te.iloc[:2000])`. Base value `base = float(np.ravel(ex.expected_value)[0])`.
2. Test additivity: the largest gap between `base + sv.sum(1)` and `model.decision_function(X_te.iloc[:2000])`.
3. `importance`: each column's mean |SHAP|, as a pandas Series with the column names, sorted from largest to smallest.
4. Draw `shap.dependence_plot(importance.index[0], sv, X_te.iloc[:2000])` for the most important variable and interpret it in one sentence. If you like, also draw one customer's waterfall plot: `shap.plots.waterfall(ex(X_te.iloc[:2000])[0])`.

**Audit questions:** On which scale are the contributions (log-odds or probability)? Does Gemini's interpretation say “the model relies on this variable” or “this variable causes default”?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(model, X_te, sv, base, importance)

## Your audit notes

- How did Gemini choose the number of trees? Did you need to change anything?
- What did boosting gain over logistic regression? Would this gap change a decision?
- How would you explain the SHAP results to a credit committee? (most important variables, log-odds scale, not causality, fairness)